In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
import uuid

In [3]:
from typing import List
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

In [4]:
import os
import time
from crewai import Agent, Task, Crew, Process
from crewai_tools import tool

In [5]:
# Define OpenAI environment variables
os.environ["OPENAI_API_BASE"] = 'https://api.groq.com/openai/v1'
os.environ["OPENAI_MODEL_NAME"] = 'llama3-70b-8192'
os.environ["OPENAI_API_KEY"] = "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"

In [6]:
df = pd.read_csv('/home/jovyan/anaconda/datasets/flights_dataset.csv')

In [8]:
df

,id,flight,source_city,destination_city,price,departure_date,arrival_date,class,airline
0,2820,G8-1404,Delhi,Mumbai,4793,2024-07-19 02:29:25,2024-07-19 09:34:13,Economy,GO_FIRST
1,3531,G8-199,Delhi,Mumbai,5954,2024-07-19 18:23:03,2024-07-20 02:57:51,Economy,GO_FIRST
2,3543,AI-479,Delhi,Mumbai,6724,2024-07-19 04:42:09,2024-07-19 22:27:09,Economy,Air_India
3,5959,G8-266,Delhi,Mumbai,10609,2024-07-19 02:49:55,2024-07-19 14:34:55,Economy,GO_FIRST
4,6375,UK-727,Delhi,Mumbai,7575,2024-07-19 16:03:58,2024-07-20 08:33:58,Economy,Vistara
...,...,...,...,...,...,...,...,...,...
551,291571,AI-440,Chennai,Mumbai,49553,2024-07-19 22:48:35,2024-07-20 14:48:35,Business,Air_India
552,293084,UK-832,Chennai,Mumbai,49553,2024-07-19 09:49:11,2024-07-19 19:23:59,Business,Vistara
553,295339,AI-440,Chennai,Bangalore,60260,2024-07-19 21:15:48,2024-07-20 11:11:00,Business,Air_India
554,297631,UK-822,Chennai,Kolkata,54896,2024-07-19 12:35:17,2024-07-19 19:40:05,Business,Vistara


## functions

In [8]:
def write_result_to_file(result, file_path):
    
    with open(file_path, 'w') as file:
            file.write(result)
    print(f"Sonuç başarıyla {file_path} dosyasına yazıldı.")

In [9]:
def create_analyst_agent():
    analyst = Agent(
                    role='Veri Analisti',
                    goal='Verilen veri kümesi hakkında detaylı analiz yap ve veri ile ilgili bilgileri araştır.',
                    backstory="""
                    Sen uzman bir araştırmacısın. 
                    Araştırma yaparken İngilizce kullanıyorsun.
                    Teknik konulara hakimsin. 
                    Belirli bir veri kümesi hakkında derinlemesine bilgi toplaman gerekiyor.
                    """,
                    verbose=False,
                    allow_delegation=False
                )
      
    return analyst

In [10]:
def create_write_agent():
    writer = Agent(
                    role='Raporlayıcı',
                    goal='Araştırmacının sağladığı bilgileri kullanarak en yüksek geliri elde edebilmek için değişkenler ve fiyatın ilişkisinin incelenmesi ve ortalama fiyat ne kadar  olamalı ,nasıl belirlenmelidir açıkla.',
                    backstory="""
                    Sen uzman bir fiyat raporlayıcısısın. 
                    Teknik konulara hakimsin. 
                    Teknik ve bilgilendirici içerik üretmek senin uzmanlık alanın.
                    Olabildiğince kısa ve anlaşılır cümleler kuruyorsun.
                    Eğer kullanacağın bilgilerde eksikler varsa, araştırmacıya soruyorsun.
                    """,
                    verbose=False,
                    allow_delegation=False 
                )
    return writer

In [11]:
def create_research_task(analyst,df):
    research_task = Task(
                        description=f"""Verilen veri kümesi ({df}) hakkında detaylı analiz yap ve konuyu araştır.""",
                        expected_output='verilen değişkenler ve fiyatlar arasındaki ilişkinin yorumlanması',
                        agent=analyst,
                        verbose=False
                    )
        
    return research_task

In [12]:
def create_write_task(writer,expected_output):
    write_task = Task(
                    description=f"""Araştırmacının sağladığı bilgileri kullanarak bilgilendirici ve uzman diliyle yazılmış bir açıklama metini yaz. 
                    Metin, her değişken arasındaki ilişki için olası optimal fiyat aralıklarını vermeli ve sebeplerini açıklamalıdır.
                    """,
                    expected_output=expected_output,
                    agent=writer,
                    verbose=False
                )
        
    return write_task

## split data

### 1) Business vs Economy (based on airlines)

In [9]:
columns_1 = ['id', 'airline','class','price']

In [10]:
df_1 = df[columns_1]

In [11]:
df_1.head()

,id,airline,class,price
0,2820,GO_FIRST,Economy,4793
1,3531,GO_FIRST,Economy,5954
2,3543,Air_India,Economy,6724
3,5959,GO_FIRST,Economy,10609
4,6375,Vistara,Economy,7575


In [16]:
analyst = create_analyst_agent()

In [17]:
writer = create_write_agent()

In [18]:
research_task = create_research_task(analyst,df)

In [19]:
expected_output="En yüksek kar için uçuş sınıfı ve airline baz alınıp parametrelere bakarak optimal fiyat aralıklarını belirlemelidir ve nasıl belirlendiği açıklanmalıdır."

In [20]:
write_task = create_write_task(writer,expected_output)

In [ ]:
crew = Crew(
            agents=[analyst, writer],
            tasks=[research_task, write_task],
            verbose=0,
            process=Process.sequential
           )

result = crew.kickoff()

In [ ]:
result

In [ ]:
write_result_to_file(result, "sonuç1.txt")

### 2) Business vs Economy (based on duration)

In [21]:
columns_2 = ['id', 'departure_date','arrival_date','class','price']

In [22]:
df_2 = df[columns_2]

In [23]:
df_2.head()

,id,departure_date,arrival_date,class,price
0,2820,2024-07-19 02:29:25,2024-07-19 09:34:13,Economy,4793
1,3531,2024-07-19 18:23:03,2024-07-20 02:57:51,Economy,5954
2,3543,2024-07-19 04:42:09,2024-07-19 22:27:09,Economy,6724
3,5959,2024-07-19 02:49:55,2024-07-19 14:34:55,Economy,10609
4,6375,2024-07-19 16:03:58,2024-07-20 08:33:58,Economy,7575


In [25]:
analyst = create_analyst_agent()
writer = create_write_agent()

In [26]:
research_task = create_research_task(analyst,df)

In [27]:
expected_output = "En yüksek kar için uçuş sınıfı ve duration baz alınıp parametrelere bakarak optimal fiyat aralıklarını belirlemelidir ve nasıl belirlendiği açıklanmalıdır."

In [28]:
write_task = create_write_task(writer,expected_output)

In [ ]:
crew = Crew(
            agents=[analyst, writer],
            tasks=[research_task, write_task],
            verbose=0,
            process=Process.sequential
           )

result = crew.kickoff()

In [ ]:
write_result_to_file(result, "sonuç2.txt")

## 3)Departure Date

In [14]:
columns_3 = ['id', 'departure_date','price']

In [15]:
df_3 = df[columns_3]

In [16]:
df_3.head()

,id,departure_date,price
0,2820,2024-07-19 02:29:25,4793
1,3531,2024-07-19 18:23:03,5954
2,3543,2024-07-19 04:42:09,6724
3,5959,2024-07-19 02:49:55,10609
4,6375,2024-07-19 16:03:58,7575


In [32]:
analyst = create_analyst_agent()
writer = create_write_agent()

In [33]:
research_task = create_research_task(analyst,df)

In [34]:
expected_output="En yüksek kar için uçuş departure date ve fiyatlarbaz alınıp parametrelere bakarak optimal fiyat aralıklarını belirlemelidir ve nasıl belirlendiği açıklanmalıdır."

In [35]:
write_task = create_write_task(writer,expected_output)

In [21]:
crew = Crew(
    agents=[analyst, writer],
    tasks=[research_task, write_task],
    verbose=0,
    process=Process.sequential  
)


result = crew.kickoff()

In [ ]:
write_result_to_file(result, "sonuç3.txt")

In [ ]:
read_text_from_file("sonuç3.txt")

## Embedding and Storage

In [24]:
def vectorize_dataframe(df, text_columns):
    """
    Embeds specified text columns in the DataFrame.

    Parameters:
    - df (pd.DataFrame): DataFrame containing text columns to embed.
    - text_columns (list): List of column names in the DataFrame to be embedded.
    """

    model = SentenceTransformer('all-MiniLM-L6-v2')

    for column in text_columns:
        df[f'{column}_embedding'] = df[column].apply(lambda x: model.encode(x).tolist())

    return df


In [19]:
text_columns = ['class','airline']
vectorize_dataframe(df_1, text_columns)

,id,airline,class,price,class_embedding,airline_embedding
0,2820,GO_FIRST,Economy,4793,"[-0.03201739862561226, 0.045857299119234085, -...","[-0.01079344842582941, -0.03127659857273102, -..."
1,3531,GO_FIRST,Economy,5954,"[-0.03201739862561226, 0.045857299119234085, -...","[-0.01079344842582941, -0.03127659857273102, -..."
2,3543,Air_India,Economy,6724,"[-0.03201739862561226, 0.045857299119234085, -...","[-0.01028473675251007, -0.013751591555774212, ..."
3,5959,GO_FIRST,Economy,10609,"[-0.03201739862561226, 0.045857299119234085, -...","[-0.01079344842582941, -0.03127659857273102, -..."
4,6375,Vistara,Economy,7575,"[-0.03201739862561226, 0.045857299119234085, -...","[-0.05844203010201454, 0.03073136880993843, -0..."
...,...,...,...,...,...,...
551,291571,Air_India,Business,49553,"[-0.006723230704665184, 0.05950081720948219, -...","[-0.01028473675251007, -0.013751591555774212, ..."
552,293084,Vistara,Business,49553,"[-0.006723230704665184, 0.05950081720948219, -...","[-0.05844203010201454, 0.03073136880993843, -0..."
553,295339,Air_India,Business,60260,"[-0.006723230704665184, 0.05950081720948219, -...","[-0.01028473675251007, -0.013751591555774212, ..."
554,297631,Vistara,Business,54896,"[-0.006723230704665184, 0.05950081720948219, -...","[-0.05844203010201454, 0.03073136880993843, -0..."


In [36]:
def load_into_qdrant(df, text_columns, collection_name):
    """
    Load points into Qdrant collection with embeddings and payloads.
    """
    # Initialize Qdrant client (replace with your actual client config)
    qdrant = QdrantClient(url="http://eager_cerf:6333")
    
    points = []
    
    # Iterate through DataFrame rows
    for _, row in df.iterrows():
        # Flatten the list of embeddings into a single vector
        vector = [value for col in text_columns for value in row[f'{col}_embedding']]
        
        # Create a unique ID for each point
        point_id = str(uuid.uuid4())
        
        # Create a point with embeddings and payload (text)
        point = models.PointStruct(
            id=point_id,
            vector=vector,
            payload={"description": row[text_columns[0]]}  # Using first text column for description
        )
        
        points.append(point)
    
    # Upload points to Qdrant in batches
    qdrant.upload_points(
        collection_name=collection_name,
        points=points,
        batch_size=64,   
        parallel=1,      
        max_retries=3,   
        wait=False       
    )
    print(f"Data successfully indexed in collection '{collection_name}'.")

Data successfully indexed in collection 'text_embeddings'.


In [ ]:
text_columns = ['class', 'airline']
collection_name = 'text_embeddings'

df = vectorize_dataframe(df_1, text_columns)
load_points(df, text_columns, collection_name)